# Проверка целостности feature-датасетов в S3

Ноутбук проверяет дневные parquet-партиции `unified_dataset` и `hmm_dataset`:

- есть ли все дневные партиции в заданном или фактическом диапазоне дат;
- непрерывен ли минутный `timestamp` внутри партиций и между соседними днями;
- нет ли дубликатов, невалидных timestamp или строк не из своего `date=...` раздела;
- нет ли `NaN`/`None` в середине ряда по каждой переменной;
- нет ли `inf`/`-inf` в числовых колонках.

Для лагов и rolling/window-признаков разрешены пустые значения только до первого валидного значения в колонке. Любой пропуск после первого валидного значения считается разрывом.

In [ ]:
from __future__ import annotations

import io
import json
import os
import re
from dataclasses import dataclass, field
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
from botocore.exceptions import ClientError
from dotenv import load_dotenv
from IPython.display import display

In [ ]:
# Основной конфиг проверки.
BUCKET = os.getenv("YC_BUCKET", "binance-data-downloader")
SYMBOL = "ADAUSDT"
INTERVAL = "1m"
EXPECTED_FREQ = "1min"

# Если оставить None, ноутбук возьмет фактический min/max из найденных партиций.
# Укажите строки YYYY-MM-DD, если нужно зафиксировать контрактный диапазон.
START_DATE = None
END_DATE = None

DATASETS = {
    "unified_dataset": {
        "prefix": "features/unified_dataset",
        "symbol": SYMBOL,
        "interval": INTERVAL,
        "expected_rows_per_full_day": 1440,
    },
    "hmm_dataset": {
        "prefix": "features/hmm_dataset",
        "symbol": SYMBOL,
        "interval": INTERVAL,
        "expected_rows_per_full_day": 1440,
    },
}

# Ограничитель числа примеров в отчетах, чтобы ноутбук не раздувался на больших разрывах.
MAX_EXAMPLES_PER_COLUMN = 20
MAX_EXAMPLES_PER_DATASET = 2000

REPORT_DIR = Path("analysis/data_integrity_reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
load_dotenv(".env")

required_env = ("YC_ENDPOINT", "YC_REGION", "YC_ACCESS_KEY_ID", "YC_SECRET_ACCESS_KEY")
missing_env = [name for name in required_env if not os.getenv(name)]
if missing_env:
    raise RuntimeError(f"Missing S3 environment variables: {', '.join(missing_env)}")

s3 = boto3.client(
    "s3",
    endpoint_url=os.getenv("YC_ENDPOINT"),
    region_name=os.getenv("YC_REGION"),
    aws_access_key_id=os.getenv("YC_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("YC_SECRET_ACCESS_KEY"),
)

In [ ]:
DATE_RE = re.compile(r"/date=(\d{4}-\d{2}-\d{2})/data\.parquet$")


def dataset_prefix(config: dict) -> str:
    return (
        f"{config['prefix'].strip('/')}/symbol={config['symbol']}/"
        f"interval={config['interval']}/"
    )


def partition_key(config: dict, date: str) -> str:
    return f"{dataset_prefix(config)}date={date}/data.parquet"


def list_dataset_partitions(bucket: str, config: dict) -> pd.DataFrame:
    prefix = dataset_prefix(config)
    rows = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            match = DATE_RE.search(f"/{key}")
            if not match:
                continue
            rows.append(
                {
                    "date": match.group(1),
                    "key": key,
                    "size_bytes": obj.get("Size", 0),
                    "last_modified": obj.get("LastModified"),
                }
            )
    return pd.DataFrame(rows).sort_values("date").reset_index(drop=True)


def read_s3_parquet(bucket: str, key: str) -> pd.DataFrame:
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
    return pd.read_parquet(io.BytesIO(body))


def expected_dates(found_dates: list[str]) -> pd.DatetimeIndex:
    if START_DATE is not None:
        start = pd.Timestamp(START_DATE)
    else:
        start = pd.Timestamp(min(found_dates))
    if END_DATE is not None:
        end = pd.Timestamp(END_DATE)
    else:
        end = pd.Timestamp(max(found_dates))
    return pd.date_range(start, end, freq="D")


def compact_timestamp_runs(timestamps: pd.Series | pd.DatetimeIndex, freq: str = EXPECTED_FREQ) -> list[dict]:
    if len(timestamps) == 0:
        return []
    idx = pd.DatetimeIndex(pd.to_datetime(timestamps, utc=True)).sort_values().unique()
    expected = pd.date_range(idx.min(), idx.max(), freq=freq)
    missing = expected.difference(idx)
    if len(missing) == 0:
        return []
    breaks = missing.to_series().diff().ne(pd.Timedelta(freq)).cumsum().to_numpy()
    result = []
    for group_id in np.unique(breaks):
        group = missing[breaks == group_id]
        result.append(
            {
                "start": group[0].isoformat(),
                "end": group[-1].isoformat(),
                "missing_rows": len(group),
            }
        )
    return result


def compact_null_runs(timestamp: pd.Series, is_null: pd.Series, limit: int) -> list[dict]:
    missing_ts = pd.DatetimeIndex(timestamp.loc[is_null].sort_values().unique())
    if len(missing_ts) == 0:
        return []
    breaks = missing_ts.to_series().diff().ne(pd.Timedelta(EXPECTED_FREQ)).cumsum().to_numpy()
    result = []
    for group_id in np.unique(breaks):
        group = missing_ts[breaks == group_id]
        result.append(
            {
                "start": group[0].isoformat(),
                "end": group[-1].isoformat(),
                "rows": len(group),
            }
        )
        if len(result) >= limit:
            break
    return result

In [ ]:
@dataclass
class ColumnState:
    seen_valid: bool = False
    first_valid_ts: pd.Timestamp | None = None
    first_gap_ts: pd.Timestamp | None = None
    gap_rows: int = 0
    examples: list[dict] = field(default_factory=list)
    leading_null_rows: int = 0
    total_rows: int = 0
    non_null_rows: int = 0


def update_null_state(states: dict[str, ColumnState], frame: pd.DataFrame, dataset_name: str, date: str) -> list[dict]:
    timestamp = frame["timestamp"]
    issues = []
    for column in frame.columns:
        if column == "timestamp":
            continue
        state = states.setdefault(column, ColumnState())
        values = frame[column]
        is_null = values.isna()
        state.total_rows += len(values)
        state.non_null_rows += int((~is_null).sum())

        if not state.seen_valid:
            valid_positions = np.flatnonzero((~is_null).to_numpy())
            if len(valid_positions) == 0:
                state.leading_null_rows += len(values)
                continue
            first_pos = int(valid_positions[0])
            state.leading_null_rows += int(is_null.iloc[:first_pos].sum())
            state.seen_valid = True
            state.first_valid_ts = timestamp.iloc[first_pos]
            check_mask = is_null.copy()
            check_mask.iloc[: first_pos + 1] = False
        else:
            check_mask = is_null

        if check_mask.any():
            state.gap_rows += int(check_mask.sum())
            if state.first_gap_ts is None:
                state.first_gap_ts = timestamp.loc[check_mask].iloc[0]
            if len(state.examples) < MAX_EXAMPLES_PER_COLUMN:
                remaining = MAX_EXAMPLES_PER_COLUMN - len(state.examples)
                state.examples.extend(compact_null_runs(timestamp, check_mask, remaining))
            issues.append(
                {
                    "dataset": dataset_name,
                    "date": date,
                    "column": column,
                    "gap_rows_in_partition": int(check_mask.sum()),
                    "first_gap_in_partition": timestamp.loc[check_mask].iloc[0].isoformat(),
                }
            )
    return issues


def summarize_null_states(dataset_name: str, states: dict[str, ColumnState]) -> pd.DataFrame:
    rows = []
    for column, state in sorted(states.items()):
        rows.append(
            {
                "dataset": dataset_name,
                "column": column,
                "status": "ok" if state.seen_valid and state.gap_rows == 0 else "FAIL",
                "total_rows": state.total_rows,
                "non_null_rows": state.non_null_rows,
                "leading_null_rows_allowed": state.leading_null_rows,
                "gap_rows_after_first_valid": state.gap_rows,
                "first_valid_ts": None if state.first_valid_ts is None else state.first_valid_ts.isoformat(),
                "first_gap_ts": None if state.first_gap_ts is None else state.first_gap_ts.isoformat(),
                "gap_examples": json.dumps(state.examples, ensure_ascii=False),
            }
        )
    return pd.DataFrame(rows)

In [ ]:
def check_partition_frame(frame: pd.DataFrame, dataset_name: str, date: str, expected_rows: int | None) -> tuple[dict, list[dict], list[dict]]:
    partition_issues = []
    finite_issues = []

    if "timestamp" not in frame.columns:
        raise ValueError(f"{dataset_name} {date}: missing timestamp column")

    frame = frame.copy()
    frame["timestamp"] = pd.to_datetime(frame["timestamp"], utc=True, errors="coerce")
    invalid_ts = int(frame["timestamp"].isna().sum())
    if invalid_ts:
        partition_issues.append({"issue": "invalid_timestamp", "rows": invalid_ts})

    valid_ts = frame.dropna(subset=["timestamp"]).sort_values("timestamp")
    duplicate_ts = int(valid_ts["timestamp"].duplicated().sum())
    if duplicate_ts:
        partition_issues.append({"issue": "duplicate_timestamp", "rows": duplicate_ts})

    day_start = pd.Timestamp(date, tz="UTC")
    day_end = day_start + pd.Timedelta(days=1)
    outside_day = int((~valid_ts["timestamp"].between(day_start, day_end, inclusive="left")).sum())
    if outside_day:
        partition_issues.append({"issue": "timestamp_outside_partition_day", "rows": outside_day})

    missing_runs = compact_timestamp_runs(valid_ts["timestamp"])
    for run in missing_runs[:20]:
        partition_issues.append({"issue": "timestamp_gap_inside_partition", **run})

    if expected_rows is not None and len(frame) != expected_rows:
        partition_issues.append(
            {"issue": "unexpected_row_count", "expected_rows": expected_rows, "actual_rows": len(frame)}
        )

    numeric = frame.select_dtypes(include=[np.number])
    for column in numeric.columns:
        values = numeric[column].to_numpy(dtype="float64", copy=False)
        bad = np.isinf(values)
        if bad.any():
            finite_issues.append(
                {
                    "dataset": dataset_name,
                    "date": date,
                    "column": column,
                    "inf_rows": int(bad.sum()),
                }
            )

    summary = {
        "dataset": dataset_name,
        "date": date,
        "rows": len(frame),
        "columns": len(frame.columns),
        "timestamp_min": None if valid_ts.empty else valid_ts["timestamp"].min().isoformat(),
        "timestamp_max": None if valid_ts.empty else valid_ts["timestamp"].max().isoformat(),
        "issues": len(partition_issues),
    }
    return summary, partition_issues, finite_issues

In [ ]:
def audit_dataset(dataset_name: str, config: dict) -> dict[str, pd.DataFrame]:
    print(f"\n=== {dataset_name} ===")
    inventory = list_dataset_partitions(BUCKET, config)
    if inventory.empty:
        raise FileNotFoundError(f"No parquet partitions found at s3://{BUCKET}/{dataset_prefix(config)}")

    dates_expected = expected_dates(inventory["date"].tolist())
    found_dates = set(inventory["date"])
    missing_dates = [day.strftime("%Y-%m-%d") for day in dates_expected if day.strftime("%Y-%m-%d") not in found_dates]
    missing_partitions = pd.DataFrame(
        [{"dataset": dataset_name, "date": date, "issue": "missing_partition"} for date in missing_dates]
    )

    partition_rows = []
    partition_issue_rows = []
    finite_issue_rows = []
    null_issue_rows = []
    timestamp_continuity_rows = []
    column_states: dict[str, ColumnState] = {}
    previous_ts: pd.Timestamp | None = None

    selected_inventory = inventory[inventory["date"].isin([d.strftime("%Y-%m-%d") for d in dates_expected])]
    total = len(selected_inventory)
    for pos, row in enumerate(selected_inventory.itertuples(index=False), start=1):
        date = row.date
        key = row.key
        if pos == 1 or pos % 50 == 0 or pos == total:
            print(f"{dataset_name}: {pos}/{total} {date}")

        frame = read_s3_parquet(BUCKET, key)
        summary, partition_issues, finite_issues = check_partition_frame(
            frame,
            dataset_name,
            date,
            config.get("expected_rows_per_full_day"),
        )
        partition_rows.append({**summary, "key": key, "size_bytes": row.size_bytes})
        partition_issue_rows.extend({"dataset": dataset_name, "date": date, **issue} for issue in partition_issues)
        finite_issue_rows.extend(finite_issues)

        frame = frame.copy()
        frame["timestamp"] = pd.to_datetime(frame["timestamp"], utc=True, errors="coerce")
        frame = frame.dropna(subset=["timestamp"]).sort_values("timestamp").drop_duplicates("timestamp")

        if not frame.empty:
            current_min = frame["timestamp"].iloc[0]
            current_max = frame["timestamp"].iloc[-1]
            if previous_ts is not None and current_min != previous_ts + pd.Timedelta(EXPECTED_FREQ):
                expected_next = previous_ts + pd.Timedelta(EXPECTED_FREQ)
                gap_end = current_min - pd.Timedelta(EXPECTED_FREQ)
                missing = max(0, int((current_min - expected_next) / pd.Timedelta(EXPECTED_FREQ)))
                timestamp_continuity_rows.append(
                    {
                        "dataset": dataset_name,
                        "previous_timestamp": previous_ts.isoformat(),
                        "next_timestamp": current_min.isoformat(),
                        "expected_next_timestamp": expected_next.isoformat(),
                        "gap_end": gap_end.isoformat(),
                        "missing_rows": missing,
                    }
                )
            previous_ts = current_max

        null_issue_rows.extend(update_null_state(column_states, frame, dataset_name, date))

    null_summary = summarize_null_states(dataset_name, column_states)
    return {
        "inventory": inventory,
        "missing_partitions": missing_partitions,
        "partition_report": pd.DataFrame(partition_rows),
        "partition_issues": pd.DataFrame(partition_issue_rows),
        "timestamp_continuity_issues": pd.DataFrame(timestamp_continuity_rows),
        "null_gap_partition_issues": pd.DataFrame(null_issue_rows[:MAX_EXAMPLES_PER_DATASET]),
        "null_gap_summary": null_summary,
        "finite_issues": pd.DataFrame(finite_issue_rows),
    }

In [ ]:
reports = {name: audit_dataset(name, config) for name, config in DATASETS.items()}

In [ ]:
def concat_report(report_name: str) -> pd.DataFrame:
    frames = [dataset_reports[report_name] for dataset_reports in reports.values() if not dataset_reports[report_name].empty]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


combined = {
    "missing_partitions": concat_report("missing_partitions"),
    "partition_report": concat_report("partition_report"),
    "partition_issues": concat_report("partition_issues"),
    "timestamp_continuity_issues": concat_report("timestamp_continuity_issues"),
    "null_gap_partition_issues": concat_report("null_gap_partition_issues"),
    "null_gap_summary": concat_report("null_gap_summary"),
    "finite_issues": concat_report("finite_issues"),
}

summary_rows = []
for dataset_name, dataset_reports in reports.items():
    null_summary = dataset_reports["null_gap_summary"]
    failing_columns = 0 if null_summary.empty else int((null_summary["status"] != "ok").sum())
    summary_rows.append(
        {
            "dataset": dataset_name,
            "partitions": len(dataset_reports["inventory"]),
            "missing_partitions": len(dataset_reports["missing_partitions"]),
            "partition_issues": len(dataset_reports["partition_issues"]),
            "timestamp_continuity_issues": len(dataset_reports["timestamp_continuity_issues"]),
            "columns_with_null_gaps_or_all_null": failing_columns,
            "finite_issues": len(dataset_reports["finite_issues"]),
        }
    )

audit_summary = pd.DataFrame(summary_rows)
display(audit_summary)

In [ ]:
for name, frame in combined.items():
    print(f"\n{name}: {len(frame)}")
    if not frame.empty:
        display(frame.head(50))

In [ ]:
for name, frame in {"audit_summary": audit_summary, **combined}.items():
    path = REPORT_DIR / f"{name}.csv"
    frame.to_csv(path, index=False)
    print(path.resolve())

failed = audit_summary.drop(columns=["partitions"]).sum(numeric_only=True).sum() > 0
if failed:
    raise AssertionError("S3 dataset integrity audit found issues. See tables above and CSV reports.")

print("OK: integrity audit passed for all configured datasets.")